# Explainable Credit Risk Copilot — Data Preparation
## Step 1: Convert `5year.arff` → `borrower_financial_features.parquet`

**Source:** Polish Bankruptcy Dataset (5th Year — 1-year forecasting horizon)  
**Volume:** `demodatabrciks123.creditrisk.raw_file/5year.arff`  
**Output (exploration):** `demodatabrciks123.creditrisk.raw_file/structured/borrower_financial_features_exploration.parquet`  
**Output (model splits):** `demodatabrciks123.creditrisk.raw_file/structured/splits/{train,validation,test}.parquet`

| Property | Value |
|---|---|
| Instances | 5,910 |
| Features | 64 financial ratios |
| Bankrupt | 410 (6.9%) |
| Non-bankrupt | 5,500 (93.1%) |
| Target column | `bankruptcy_label` (1 = bankrupt, 0 = solvent) |
| Missing values (exploration) | Replaced with column median — exploration file only |
| Missing values (splits) | Kept as-is; imputer must be fit on `train.parquet` only |

In [0]:
# =============================================================
# CONFIGURATION 
# =============================================================
SOURCE_ARFF   = "/Volumes/demodatabrciks123/creditrisk/raw_file/5year.arff"
OUTPUT_DIR    = "/Volumes/demodatabrciks123/creditrisk/raw_file/structured"
OUTPUT_FILE   = f"{OUTPUT_DIR}/borrower_financial_features_exploration.parquet"
FORECAST_YEAR = 5   # dataset year (1–5); 5 = 1-year horizon

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Rename previously written file (if it exists under the old name)
_old_file = f"{OUTPUT_DIR}/borrower_financial_features.parquet"
if os.path.exists(_old_file) and not os.path.exists(OUTPUT_FILE):
    os.rename(_old_file, OUTPUT_FILE)
    print(f"Renamed: borrower_financial_features.parquet → borrower_financial_features_exploration.parquet")

print(f"Source      : {SOURCE_ARFF}")
print(f"Exploration : {OUTPUT_FILE}")
print(f"Splits dir  : {OUTPUT_DIR}/structured/splits/")

In [0]:
# =============================================================
# PARSE ARFF — header attributes + data rows
# Missing values in ARFF are encoded as '?'; we keep them as NaN
# =============================================================
import re
import pandas as pd
import numpy as np

def parse_arff(path: str) -> pd.DataFrame:
    """Minimal ARFF parser — handles numeric attributes and missing '?' values."""
    attributes = []
    data_rows   = []
    in_data     = False

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("%"):
                continue

            lower = line.lower()
            if lower.startswith("@attribute"):
                parts = line.split()
                attributes.append(parts[1].strip("'\""))
            elif lower == "@data":
                in_data = True
            elif in_data:
                values = []
                for v in line.split(","):
                    v = v.strip()
                    values.append(np.nan if v == "?" else v)
                data_rows.append(values)

    df = pd.DataFrame(data_rows, columns=attributes)
    # Cast numeric columns; class stays as-is then we rename
    for col in df.columns:
        if col.lower() != "class":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["class"] = df["class"].astype(int)
    return df

raw_df = parse_arff(SOURCE_ARFF)
print(f"Shape      : {raw_df.shape}")
print(f"Columns    : {list(raw_df.columns[:5])} ... {list(raw_df.columns[-3:])}")
print(f"Missing %  : {raw_df.isnull().mean().mean()*100:.2f}% overall")
raw_df.head(3)

In [0]:
# =============================================================
# COLUMN MAPPING — Attr1..Attr64 -> readable financial ratio names
# Source: UCI Polish Bankruptcy Dataset documentation
# =============================================================
COLUMN_NAMES = [
    "net_profit_to_total_assets",                  # Attr1
    "total_liabilities_to_total_assets",           # Attr2
    "working_capital_to_total_assets",             # Attr3
    "current_assets_to_short_term_liabilities",    # Attr4
    "cash_flow_coverage_days",                     # Attr5  [(cash+ST_sec+receivables-ST_liab)/(op_exp-depr)]*365
    "retained_earnings_to_total_assets",           # Attr6
    "ebit_to_total_assets",                        # Attr7
    "book_value_equity_to_total_liabilities",      # Attr8
    "sales_to_total_assets",                       # Attr9  asset turnover
    "equity_to_total_assets",                      # Attr10
    "gross_profit_plus_fin_exp_to_total_assets",   # Attr11
    "gross_profit_to_short_term_liabilities",      # Attr12
    "gross_profit_plus_depr_to_sales",             # Attr13
    "gross_profit_plus_interest_to_total_assets",  # Attr14
    "total_liabilities_days_to_gross_profit",      # Attr15  (total_liab*365)/(gross_profit+depr)
    "gross_profit_plus_depr_to_total_liabilities", # Attr16
    "total_assets_to_total_liabilities",           # Attr17
    "gross_profit_to_total_assets",                # Attr18
    "gross_profit_margin",                         # Attr19  gross_profit/sales
    "inventory_days",                              # Attr20  (inventory*365)/sales
    "sales_growth_yoy",                            # Attr21  sales(n)/sales(n-1)
    "operating_profit_to_total_assets",            # Attr22
    "net_profit_margin",                           # Attr23  net_profit/sales
    "gross_profit_3yr_avg_to_total_assets",        # Attr24
    "retained_capital_to_total_assets",            # Attr25  (equity - share_capital)/TA
    "net_profit_plus_depr_to_total_liabilities",   # Attr26
    "operating_profit_to_financial_expenses",      # Attr27
    "working_capital_to_fixed_assets",             # Attr28
    "log_total_assets",                            # Attr29
    "net_liabilities_to_sales",                    # Attr30  (total_liab - cash)/sales
    "gross_profit_plus_interest_to_sales",         # Attr31
    "current_liabilities_days",                    # Attr32  (current_liab*365)/cost_of_sales
    "operating_expenses_to_short_term_liabilities",# Attr33
    "operating_expenses_to_total_liabilities",     # Attr34
    "profit_on_sales_to_total_assets",             # Attr35
    "total_sales_to_total_assets",                 # Attr36
    "current_assets_ex_inventory_to_lt_liabilities",# Attr37
    "constant_capital_to_total_assets",            # Attr38
    "profit_on_sales_margin",                      # Attr39  profit_on_sales/sales
    "quick_ratio_ex_receivables",                  # Attr40  (current_assets-inv-rec)/ST_liab
    "debt_coverage_days",                          # Attr41  total_liab/((op_profit+depr)*(12/365))
    "operating_profit_margin",                     # Attr42  op_profit/sales
    "receivables_plus_inventory_turnover_days",    # Attr43
    "receivable_days",                             # Attr44  (receivables*365)/sales
    "net_profit_to_inventory",                     # Attr45
    "quick_ratio",                                 # Attr46  (current_assets-inventory)/ST_liab
    "inventory_days_cost",                         # Attr47  (inventory*365)/cost_of_products_sold
    "ebitda_to_total_assets",                      # Attr48
    "ebitda_margin",                               # Attr49  EBITDA/sales
    "current_assets_to_total_liabilities",         # Attr50
    "short_term_liabilities_to_total_assets",      # Attr51
    "short_term_liabilities_days_cost",            # Attr52  (ST_liab*365)/cost_of_products_sold
    "equity_to_fixed_assets",                      # Attr53
    "constant_capital_to_fixed_assets",            # Attr54
    "working_capital",                             # Attr55  absolute value
    "gross_margin",                                # Attr56  (sales - cost_of_products_sold)/sales
    "liquidity_to_operations_ratio",               # Attr57  (current_assets-inv-ST_liab)/(sales-gross_profit-depr)
    "total_costs_to_total_sales",                  # Attr58
    "long_term_liabilities_to_equity",             # Attr59
    "sales_to_inventory",                          # Attr60  inventory turnover
    "sales_to_receivables",                        # Attr61  receivables turnover
    "short_term_liabilities_days_sales",           # Attr62  (ST_liab*365)/sales
    "sales_to_short_term_liabilities",             # Attr63
    "sales_to_fixed_assets",                       # Attr64  fixed asset turnover
]

# Rename Attr1..Attr64 -> descriptive names, 'class' -> 'bankruptcy_label'
rename_map = {f"Attr{i+1}": name for i, name in enumerate(COLUMN_NAMES)}
rename_map["class"] = "bankruptcy_label"

df = raw_df.rename(columns=rename_map)
print(f"Columns renamed: {len(rename_map)}")
print(df.dtypes.value_counts().to_string())

In [0]:
# =============================================================
# DATA QUALITY
# 1. Impute missing numeric values with per-column MEDIAN
#    (robust to outliers; preserves distribution shape)
# 2. Clip extreme outliers at 1st/99th percentile per feature
# 3. Add metadata columns for lineage and model training context
# =============================================================
from sklearn.impute import SimpleImputer  # available on Databricks runtime

feature_cols = COLUMN_NAMES  # 64 ratio columns (no target)

# --- Missing value report before imputation ---
missing = df[feature_cols].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print("Top 10 columns by missing % (pre-imputation):")
print(missing_pct.sort_values(ascending=False).head(10).to_string())

# --- Median imputation ---
imputer = SimpleImputer(strategy="median")
df[feature_cols] = imputer.fit_transform(df[feature_cols])

# --- Clip extreme outliers (winsorisation at 1st / 99th pct) ---
for col in feature_cols:
    lo = df[col].quantile(0.01)
    hi = df[col].quantile(0.99)
    df[col] = df[col].clip(lo, hi)

# --- Add metadata columns ---
df.insert(0, "borrower_id",     range(1, len(df) + 1))  # synthetic stable ID
df.insert(1, "dataset_year",    FORECAST_YEAR)           # 5
df.insert(2, "forecast_horizon", 1)                       # 1-year horizon
df.insert(3, "observation_window", "2007-2013")           # per dataset description

# Move target to last column
df = df[[c for c in df.columns if c != "bankruptcy_label"] + ["bankruptcy_label"]]

print(f"\nFinal shape  : {df.shape}")
print(f"Bankrupt     : {df['bankruptcy_label'].sum()} ({df['bankruptcy_label'].mean()*100:.1f}%)")
print(f"Non-bankrupt : {(df['bankruptcy_label']==0).sum()}")
print(f"Remaining NaN: {df.isnull().sum().sum()}")
df.head(3)

In [0]:
# =============================================================
# WRITE — Exploration parquet (imputed + winsorised; NOT for model evaluation)
# Path: /Volumes/demodatabrciks123/creditrisk/raw_file/structured/borrower_financial_features_exploration.parquet
# =============================================================
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(
    table,
    OUTPUT_FILE,
    compression="snappy",
    write_statistics=True,
)

size_mb = os.path.getsize(OUTPUT_FILE) / 1_048_576
print(f"Written  : {OUTPUT_FILE}")
print(f"Size     : {size_mb:.2f} MB")
print(f"Rows     : {len(df):,}")
print(f"Columns  : {len(df.columns)} (4 metadata + 64 features + 1 target)")

In [0]:
# =============================================================
# VALIDATE — read back the parquet and confirm schema / counts
# =============================================================
validation_df = pd.read_parquet(OUTPUT_FILE)

print("=== Schema (first 10 columns) ===")
print(validation_df.dtypes.head(10).to_string())
print(f"\n=== Row & column counts ===")
print(f"  Rows    : {len(validation_df):,}")
print(f"  Columns : {len(validation_df.columns)}")

print(f"\n=== Target distribution ===")
print(validation_df["bankruptcy_label"].value_counts().rename({0: "Solvent", 1: "Bankrupt"}).to_string())

print(f"\n=== Sample feature stats (3 key ratios) ===")
check_cols = ["net_profit_to_total_assets", "total_liabilities_to_total_assets", "ebit_to_total_assets"]
print(validation_df[check_cols].describe().round(4).to_string())

print(f"\nParquet is valid. Ready for model training.")

In [0]:
# =============================================================
# STRATIFIED SPLITS — 70 / 15 / 15
#
# IMPORTANT — data-leakage-free design:
#   Start from the RENAMED data BEFORE any imputation or clipping.
#   The median imputer and winsorisation thresholds must be fitted
#   on train.parquet ONLY, then applied unchanged to validation
#   and test splits in a later notebook.
# =============================================================
from sklearn.model_selection import train_test_split

SPLIT_DIR = f"{OUTPUT_DIR}/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

# ---- Build model_df from raw renamed data (before imputation) ----
model_df = raw_df.rename(columns=rename_map).copy()

# B-format borrower_id for linking future synthetic documents
model_df.insert(
    0,
    "borrower_id",
    [f"B{i:06d}" for i in range(1, len(model_df) + 1)]
)

# Clearer target name for the modelling pipeline
model_df = model_df.rename(columns={"bankruptcy_label": "bankrupt_within_1_year"})

target_col = "bankrupt_within_1_year"

# ---- 70 / 15 / 15 stratified split ----
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df[target_col],
    random_state=42,
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[target_col],
    random_state=42,
)

# Add lineage column after splitting (no pre-split leakage)
train_df      = train_df.copy();      train_df["dataset_split"]      = "train"
validation_df = validation_df.copy(); validation_df["dataset_split"] = "validation"
test_df       = test_df.copy();       test_df["dataset_split"]       = "test"

# ---- Write splits (missing values intentionally preserved) ----
train_df.to_parquet(f"{SPLIT_DIR}/train.parquet",      index=False)
validation_df.to_parquet(f"{SPLIT_DIR}/validation.parquet", index=False)
test_df.to_parquet(f"{SPLIT_DIR}/test.parquet",        index=False)

# ---- Summary ----
print(f"Split directory : {SPLIT_DIR}")
print(f"{'Split':10s} | {'Rows':>5} | {'Bankrupt':>8} | {'Rate':>7} | {'Missing NaN':>11}")
print("-" * 55)
for name, split_df in {"train": train_df, "validation": validation_df, "test": test_df}.items():
    nan_count = split_df.isnull().sum().sum()
    print(
        f"{name:10s} | {len(split_df):5d} | "
        f"{split_df[target_col].sum():8d} | "
        f"{split_df[target_col].mean():.2%} | "
        f"{nan_count:11,d}"
    )
print("\nMissing values kept — fit imputer on train.parquet only.")

In [0]:
# =============================================================
# DEMO COHORT — 30 borrowers selected from test.parquet ONLY
#
# Groups:
#   - 10 healthy solvent   (lowest stress score)
#   - 10 stressed solvent  (highest stress score, still solvent)
#   - 10 bankrupt          (highest stress score among bankrupt)
#
# NOTE: selection_stress_score and demo_risk_group are
# portfolio-construction fields ONLY. Do not use as model
# features or treat as validated credit-risk classifications.
# =============================================================
import os
import numpy as np
import pandas as pd

TEST_FILE = f"{SPLIT_DIR}/test.parquet"

DEMO_DIR = f"{OUTPUT_DIR}/demo_cohort"
DEMO_COHORT_FILE = f"{DEMO_DIR}/demo_borrower_cohort.parquet"
BORROWER_MASTER_FILE = f"{DEMO_DIR}/borrower_master.csv"

os.makedirs(DEMO_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. Load the untouched test split
# ---------------------------------------------------------
test_df = pd.read_parquet(TEST_FILE).copy()

target_col = "bankrupt_within_1_year"

# ---------------------------------------------------------
# 2. Create a transparent stress score for cohort selection
#
# This score is ONLY for selecting varied demo companies.
# It is not the future credit-risk model.
# ---------------------------------------------------------
stress_features = {
    # Higher value = more financial stress
    "net_profit_to_total_assets": "lower_is_worse",
    "total_liabilities_to_total_assets": "higher_is_worse",
    "working_capital_to_total_assets": "lower_is_worse",
    "current_assets_to_short_term_liabilities": "lower_is_worse",
    "ebit_to_total_assets": "lower_is_worse",
}

stress_components = []

for column, direction in stress_features.items():

    percentile_rank = test_df[column].rank(
        method="average",
        pct=True,
        na_option="keep"
    )

    if direction == "lower_is_worse":
        component = 1.0 - percentile_rank
    else:
        component = percentile_rank

    # Treat missing information as neutral for cohort selection
    component = component.fillna(0.50)

    stress_components.append(component)

test_df["selection_stress_score"] = pd.concat(
    stress_components,
    axis=1
).mean(axis=1)

# ---------------------------------------------------------
# 3. Create three demo groups
# ---------------------------------------------------------
solvent_df = test_df[
    test_df[target_col] == 0
].copy()

bankrupt_df = test_df[
    test_df[target_col] == 1
].copy()

healthy_demo = (
    solvent_df
    .nsmallest(10, "selection_stress_score")
    .copy()
)

healthy_demo["demo_risk_group"] = "healthy_solvent"

stressed_demo = (
    solvent_df
    .nlargest(10, "selection_stress_score")
    .copy()
)

stressed_demo["demo_risk_group"] = "stressed_solvent"

bankrupt_demo = (
    bankrupt_df
    .nlargest(10, "selection_stress_score")
    .copy()
)

bankrupt_demo["demo_risk_group"] = "bankrupt"

demo_cohort = pd.concat(
    [
        healthy_demo,
        stressed_demo,
        bankrupt_demo
    ],
    ignore_index=True
)

# Randomize display order, but keep it reproducible
demo_cohort = demo_cohort.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ---------------------------------------------------------
# 4. Save the selected financial records
# ---------------------------------------------------------
demo_cohort.to_parquet(
    DEMO_COHORT_FILE,
    index=False
)

# ---------------------------------------------------------
# 5. Create synthetic company metadata
# ---------------------------------------------------------
company_prefixes = [
    "NorthStar", "GreenBridge", "Delta", "Nova", "Orion",
    "BluePeak", "RiverStone", "Atlas", "SilverLine", "Vertex",
    "ClearPath", "BrightCore", "Oakfield", "Evercrest", "Harbor",
    "PrimeWorks", "Westhaven", "IronGate", "Lumina", "Apex",
    "Redwood", "Skyline", "MetroCore", "TerraNova", "Summit",
    "CedarPoint", "Windmere", "Stonebridge", "Horizon", "Amber"
]

company_suffixes = [
    "Manufacturing B.V.",
    "Logistics B.V.",
    "Technology B.V.",
    "Engineering B.V.",
    "Services B.V."
]

sectors = [
    "Manufacturing",
    "Logistics",
    "Technology",
    "Engineering Services",
    "Wholesale"
]

loan_purposes = [
    "Working capital",
    "Equipment investment",
    "Business expansion",
    "Debt refinancing",
    "Warehouse modernisation"
]

rng = np.random.default_rng(42)

borrower_master = pd.DataFrame({
    "borrower_id": demo_cohort["borrower_id"],

    "company_name": [
        f"{company_prefixes[i]} "
        f"{company_suffixes[i % len(company_suffixes)]}"
        for i in range(len(demo_cohort))
    ],

    "sector": [
        sectors[i % len(sectors)]
        for i in range(len(demo_cohort))
    ],

    "country": "Netherlands",

    "employee_count": rng.integers(
        low=25,
        high=500,
        size=len(demo_cohort)
    ),

    "requested_loan_amount_eur": (
        rng.integers(
            low=100,
            high=5000,
            size=len(demo_cohort)
        ) * 1000
    ),

    "loan_purpose": [
        loan_purposes[i % len(loan_purposes)]
        for i in range(len(demo_cohort))
    ],

    "demo_risk_group": demo_cohort["demo_risk_group"],

    # Keep the real dataset label for later evaluation
    "bankrupt_within_1_year": demo_cohort[target_col],

    "metadata_is_synthetic": True  # financial ratios & bankruptcy labels are real (UCI dataset)
})

borrower_master.to_csv(
    BORROWER_MASTER_FILE,
    index=False
)

# ---------------------------------------------------------
# 6. Validate
# ---------------------------------------------------------
print(f"Demo cohort file   : {DEMO_COHORT_FILE}")
print(f"Borrower master    : {BORROWER_MASTER_FILE}")

print("\nDemo cohort distribution:")
print(
    borrower_master["demo_risk_group"]
    .value_counts()
    .to_string()
)

print("\nTarget distribution:")
print(
    borrower_master["bankrupt_within_1_year"]
    .value_counts()
    .rename({
        0: "Solvent",
        1: "Bankrupt"
    })
    .to_string()
)

print("\nDuplicate borrower IDs:")
print(borrower_master["borrower_id"].duplicated().sum())

display(borrower_master.head())

In [0]:
# =============================================================
# CLEAN MODEL SPLITS V2
#
# Permanently excludes the 30 demo-document borrowers.
# Demo borrowers were selected from the original test split
# using features and bankruptcy labels; they are therefore
# no longer a clean holdout for any future model evaluation.
#
# Output:
#   splits_v2/train.parquet
#   splits_v2/validation.parquet
#   splits_v2/test.parquet
#
# After this:
#   - retrain and calibrate using V2 train
#   - select threshold using V2 validation
#   - evaluate once using V2 test
# =============================================================

import os
import pandas as pd

from sklearn.model_selection import train_test_split

CLEAN_SPLIT_DIR = f"{OUTPUT_DIR}/splits_v2"
os.makedirs(CLEAN_SPLIT_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. Reconstruct the original unprocessed dataset
#    Same as the original split cell: rename raw_df, add
#    B-format IDs, rename the target column.
# ---------------------------------------------------------
all_model_df = raw_df.rename(columns=rename_map).copy()

all_model_df.insert(
    0,
    "borrower_id",
    [f"B{i:06d}" for i in range(1, len(all_model_df) + 1)],
)

all_model_df = all_model_df.rename(
    columns={"bankruptcy_label": "bankrupt_within_1_year"}
)

TARGET_COLUMN = "bankrupt_within_1_year"

# ---------------------------------------------------------
# 2. Read the permanent demo cohort IDs
# ---------------------------------------------------------
demo_cohort_df = pd.read_parquet(DEMO_COHORT_FILE)
demo_borrower_ids = set(demo_cohort_df["borrower_id"])

assert len(demo_borrower_ids) == 30, (
    f"Expected 30 demo borrowers, found {len(demo_borrower_ids)}"
)

# ---------------------------------------------------------
# 3. Permanently exclude demo borrowers from modelling
# ---------------------------------------------------------
clean_model_df = all_model_df[
    ~all_model_df["borrower_id"].isin(demo_borrower_ids)
].copy()

assert len(clean_model_df) == 5880, (
    f"Expected 5880 rows after exclusion, found {len(clean_model_df)}"
)
assert set(clean_model_df["borrower_id"]).isdisjoint(demo_borrower_ids)

# ---------------------------------------------------------
# 4. Create clean 70 / 15 / 15 splits
#    random_state=2026 (distinct from original split seed 42)
# ---------------------------------------------------------
train_v2, temp_v2 = train_test_split(
    clean_model_df,
    test_size=0.30,
    stratify=clean_model_df[TARGET_COLUMN],
    random_state=2026,
)

validation_v2, test_v2 = train_test_split(
    temp_v2,
    test_size=0.50,
    stratify=temp_v2[TARGET_COLUMN],
    random_state=2026,
)

train_v2      = train_v2.copy()
validation_v2 = validation_v2.copy()
test_v2       = test_v2.copy()

train_v2["dataset_split"]      = "train"
validation_v2["dataset_split"] = "validation"
test_v2["dataset_split"]       = "test"

# ---------------------------------------------------------
# 5. Validate isolation
# ---------------------------------------------------------
train_ids      = set(train_v2["borrower_id"])
validation_ids = set(validation_v2["borrower_id"])
test_ids       = set(test_v2["borrower_id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

assert train_ids.isdisjoint(demo_borrower_ids)
assert validation_ids.isdisjoint(demo_borrower_ids)
assert test_ids.isdisjoint(demo_borrower_ids)

assert (
    len(train_v2) + len(validation_v2) + len(test_v2) == 5880
)

# Missing values must remain for pipeline-based imputation
assert train_v2.isnull().sum().sum() > 0, (
    "Train split has no missing values — "
    "imputation has leaked into raw split data."
)

# ---------------------------------------------------------
# 6. Save the clean splits
# ---------------------------------------------------------
train_v2.to_parquet(
    f"{CLEAN_SPLIT_DIR}/train.parquet", index=False
)
validation_v2.to_parquet(
    f"{CLEAN_SPLIT_DIR}/validation.parquet", index=False
)
test_v2.to_parquet(
    f"{CLEAN_SPLIT_DIR}/test.parquet", index=False
)

# ---------------------------------------------------------
# 7. Summary
# ---------------------------------------------------------
print(f"Clean split directory: {CLEAN_SPLIT_DIR}")
print(f"Excluded demo IDs    : {len(demo_borrower_ids)}")

print(
    f"\n{'Split':12s} | "
    f"{'Rows':>5s} | "
    f"{'Bankrupt':>8s} | "
    f"{'Rate':>7s}"
)
print("-" * 43)

for name, split_df in {
    "train":      train_v2,
    "validation": validation_v2,
    "test":        test_v2,
}.items():
    print(
        f"{name:12s} | "
        f"{len(split_df):5d} | "
        f"{split_df[TARGET_COLUMN].sum():8d} | "
        f"{split_df[TARGET_COLUMN].mean():7.2%}"
    )

print("\nIsolation checks:")
print("  Train \u2194 validation overlap : 0")
print("  Train \u2194 test overlap       : 0")
print("  Validation \u2194 test overlap  : 0")
print("  Demo borrowers in splits : 0")

print(
    "\nTest set remains untouched: "
    f"{CLEAN_SPLIT_DIR}/test.parquet"
)

In [0]:
# =============================================================
# DOCUMENT FACTS TABLE
#
# Purpose:
# - Merge synthetic company metadata with real financial ratios
# - Create deterministic financial observations
# - Keep future bankruptcy outcome separate from documents
# =============================================================

import os
import numpy as np
import pandas as pd

DOCUMENT_FACTS_FILE = (
    f"{DEMO_DIR}/borrower_document_facts.parquet"
)

EVALUATION_LABELS_FILE = (
    f"{DEMO_DIR}/demo_evaluation_labels.parquet"
)

# ---------------------------------------------------------
# 1. Reload source files
# ---------------------------------------------------------
demo_cohort = pd.read_parquet(DEMO_COHORT_FILE)
borrower_master = pd.read_csv(BORROWER_MASTER_FILE)

# Correct the metadata flag name
if "data_is_synthetic" in borrower_master.columns:
    borrower_master = borrower_master.rename(
        columns={"data_is_synthetic": "metadata_is_synthetic"}
    )

# Save corrected master file
borrower_master.to_csv(
    BORROWER_MASTER_FILE,
    index=False
)

# ---------------------------------------------------------
# 2. Keep future outcomes in a PRIVATE evaluation table
# ---------------------------------------------------------
evaluation_labels = demo_cohort[
    [
        "borrower_id",
        "demo_risk_group",
        "bankrupt_within_1_year",
        "selection_stress_score",
    ]
].copy()

evaluation_labels.to_parquet(
    EVALUATION_LABELS_FILE,
    index=False
)

# ---------------------------------------------------------
# 3. Select only information available at assessment time
# ---------------------------------------------------------
financial_columns = [
    "borrower_id",
    "net_profit_to_total_assets",
    "total_liabilities_to_total_assets",
    "working_capital_to_total_assets",
    "current_assets_to_short_term_liabilities",
    "retained_earnings_to_total_assets",
    "ebit_to_total_assets",
    "equity_to_total_assets",
    "sales_to_total_assets",
    "operating_profit_margin",
    "short_term_liabilities_to_total_assets",
]

metadata_columns = [
    "borrower_id",
    "company_name",
    "sector",
    "country",
    "employee_count",
    "requested_loan_amount_eur",
    "loan_purpose",
    "metadata_is_synthetic",
]

document_facts = borrower_master[
    metadata_columns
].merge(
    demo_cohort[financial_columns],
    on="borrower_id",
    how="inner",
    validate="one_to_one",
)

# ---------------------------------------------------------
# 4. Add source and lineage metadata
# ---------------------------------------------------------
document_facts["financial_data_source"] = (
    "UCI Polish Companies Bankruptcy dataset"
)

document_facts["financial_features_are_synthetic"] = False
document_facts["document_content_is_synthetic"] = True
document_facts["document_generation_version"] = "1.0"

# ---------------------------------------------------------
# 5. Create objective, deterministic financial flags
#
# These are mathematical observations, not bank policy rules.
# ---------------------------------------------------------
document_facts["negative_net_profit"] = (
    document_facts["net_profit_to_total_assets"] < 0
)

document_facts["negative_ebit"] = (
    document_facts["ebit_to_total_assets"] < 0
)

document_facts["negative_working_capital"] = (
    document_facts["working_capital_to_total_assets"] < 0
)

document_facts[
    "current_assets_below_short_term_liabilities"
] = (
    document_facts[
        "current_assets_to_short_term_liabilities"
    ] < 1
)

document_facts["liabilities_exceed_assets"] = (
    document_facts[
        "total_liabilities_to_total_assets"
    ] > 1
)

document_facts["negative_equity"] = (
    document_facts["equity_to_total_assets"] < 0
)

# ---------------------------------------------------------
# 6. Helper for safe text formatting
# ---------------------------------------------------------
def format_ratio(value, decimals=3):
    if pd.isna(value):
        return "not available"
    return f"{value:.{decimals}f}"

# ---------------------------------------------------------
# 7. Create deterministic source text
#
# This text will later become part of financial_summary.pdf.
# No LLM is used here, preventing invented financial facts.
# ---------------------------------------------------------
def create_financial_summary(row):

    observations = []

    if row["negative_net_profit"]:
        observations.append(
            "The company reported a negative net-profit-to-assets ratio."
        )
    else:
        observations.append(
            "The company reported a positive net-profit-to-assets ratio."
        )

    if row["negative_ebit"]:
        observations.append(
            "EBIT relative to total assets was negative."
        )
    else:
        observations.append(
            "EBIT relative to total assets was positive."
        )

    if row["negative_working_capital"]:
        observations.append(
            "Working capital relative to total assets was negative."
        )

    if row["current_assets_below_short_term_liabilities"]:
        observations.append(
            "Current assets were lower than short-term liabilities."
        )
    else:
        observations.append(
            "Current assets exceeded short-term liabilities."
        )

    if row["liabilities_exceed_assets"]:
        observations.append(
            "Reported total liabilities exceeded total assets."
        )

    return (
        f"Company: {row['company_name']}\n"
        f"Sector: {row['sector']}\n"
        f"Country: {row['country']}\n\n"
        f"Selected financial ratios:\n"
        f"- Net profit / total assets: "
        f"{format_ratio(row['net_profit_to_total_assets'])}\n"
        f"- Total liabilities / total assets: "
        f"{format_ratio(row['total_liabilities_to_total_assets'])}\n"
        f"- Working capital / total assets: "
        f"{format_ratio(row['working_capital_to_total_assets'])}\n"
        f"- Current assets / short-term liabilities: "
        f"{format_ratio(row['current_assets_to_short_term_liabilities'])}\n"
        f"- EBIT / total assets: "
        f"{format_ratio(row['ebit_to_total_assets'])}\n"
        f"- Equity / total assets: "
        f"{format_ratio(row['equity_to_total_assets'])}\n\n"
        f"Objective observations:\n- "
        + "\n- ".join(observations)
        + "\n\n"
        "This is synthetic document content created for a "
        "credit-risk decision-support demonstration."
    )

document_facts["financial_summary_text"] = (
    document_facts.apply(
        create_financial_summary,
        axis=1
    )
)

# ---------------------------------------------------------
# 8. Explicit leakage validation
# ---------------------------------------------------------
forbidden_document_columns = {
    "bankrupt_within_1_year",
    "demo_risk_group",
    "selection_stress_score",
}

leaked_columns = forbidden_document_columns.intersection(
    document_facts.columns
)

assert not leaked_columns, (
    f"Forbidden outcome columns found: {leaked_columns}"
)

assert len(document_facts) == 30
assert document_facts["borrower_id"].is_unique
assert document_facts["company_name"].is_unique

# Ensure the generated text does not contain outcome wording
forbidden_text = [
    "bankrupt_within_1_year",
    "demo_risk_group",
    "selection_stress_score",
]

for phrase in forbidden_text:
    assert not document_facts[
        "financial_summary_text"
    ].str.contains(
        phrase,
        case=False,
        na=False
    ).any()

# ---------------------------------------------------------
# 9. Save
# ---------------------------------------------------------
document_facts.to_parquet(
    DOCUMENT_FACTS_FILE,
    index=False
)

print(f"Document facts   : {DOCUMENT_FACTS_FILE}")
print(f"Evaluation labels: {EVALUATION_LABELS_FILE}")
print(f"Rows             : {len(document_facts)}")
print(f"Unique borrowers : {document_facts['borrower_id'].nunique()}")
print("Outcome leakage  : none")

display(
    document_facts[
        [
            "borrower_id",
            "company_name",
            "negative_net_profit",
            "negative_ebit",
            "negative_working_capital",
            "current_assets_below_short_term_liabilities",
            "liabilities_exceed_assets",
        ]
    ].head()
)

In [0]:
%pip install reportlab

In [0]:
# =============================================================
# GENERATE SYNTHETIC CREDIT-RISK DOCUMENTS
#
# Creates:
#   30 loan applications
#   30 financial summaries
#   30 preliminary analyst notes
#
# Important:
# - No bankruptcy label is included.
# - No demo risk group is included.
# - No selection stress score is included.
# - All narrative content is deterministic and auditable.
# =============================================================

import os
import html
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
)

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------
DOCUMENT_FACTS_FILE = (
    f"{DEMO_DIR}/borrower_document_facts.parquet"
)

DOCUMENT_ROOT = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/documents"
)

DOCUMENT_MANIFEST_FILE = (
    f"{DEMO_DIR}/document_manifest.parquet"
)

os.makedirs(DOCUMENT_ROOT, exist_ok=True)

document_facts = pd.read_parquet(
    DOCUMENT_FACTS_FILE
).copy()

assert len(document_facts) == 30
assert document_facts["borrower_id"].is_unique

# ---------------------------------------------------------
# 2. PDF styles
# ---------------------------------------------------------
styles = getSampleStyleSheet()

styles.add(
    ParagraphStyle(
        name="DocumentTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontName="Helvetica-Bold",
        fontSize=17,
        leading=21,
        spaceAfter=14,
    )
)

styles.add(
    ParagraphStyle(
        name="SectionHeading",
        parent=styles["Heading2"],
        fontName="Helvetica-Bold",
        fontSize=12,
        leading=15,
        spaceBefore=10,
        spaceAfter=7,
    )
)

styles.add(
    ParagraphStyle(
        name="SmallText",
        parent=styles["BodyText"],
        fontSize=8,
        leading=10,
    )
)

styles.add(
    ParagraphStyle(
        name="Disclaimer",
        parent=styles["BodyText"],
        fontSize=8,
        leading=10,
        textColor=colors.grey,
        spaceBefore=12,
    )
)

# ---------------------------------------------------------
# 3. Formatting helpers
# ---------------------------------------------------------
def safe_text(value) -> str:
    if pd.isna(value):
        return "Not available"
    return html.escape(str(value))


def format_ratio(value, decimals: int = 3) -> str:
    if pd.isna(value):
        return "Not available"
    return f"{float(value):.{decimals}f}"


def format_currency(value) -> str:
    if pd.isna(value):
        return "Not available"
    return f"\u20ac{float(value):,.0f}"


def format_boolean(value) -> str:
    return "Yes" if bool(value) else "No"


def paragraph(text: str, style: str = "BodyText"):
    return Paragraph(safe_text(text), styles[style])


def create_footer(document_id: str):
    """Return a ReportLab page callback."""

    def footer(canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 8)
        canvas.drawString(
            18 * mm, 12 * mm,
            f"Document ID: {document_id}"
        )
        canvas.drawRightString(
            A4[0] - 18 * mm, 12 * mm,
            f"Page {doc.page}"
        )
        canvas.restoreState()

    return footer


def create_table(rows, column_widths=None, header=True):
    converted_rows = [[
        Paragraph(safe_text(value), styles["SmallText"])
        for value in row
    ] for row in rows]

    table = Table(
        converted_rows,
        colWidths=column_widths,
        repeatRows=1 if header else 0,
        hAlign="LEFT",
    )

    style_commands = [
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]

    if header:
        style_commands.extend([
            ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ])

    table.setStyle(TableStyle(style_commands))
    return table


# ---------------------------------------------------------
# 4. Objective observations
# ---------------------------------------------------------
def get_objective_observations(row) -> list:

    observations = []

    if bool(row["negative_net_profit"]):
        observations.append("Net profit relative to total assets is negative.")
    else:
        observations.append("Net profit relative to total assets is positive.")

    if bool(row["negative_ebit"]):
        observations.append("EBIT relative to total assets is negative.")
    else:
        observations.append("EBIT relative to total assets is positive.")

    if bool(row["negative_working_capital"]):
        observations.append("Working capital relative to total assets is negative.")
    else:
        observations.append("Working capital relative to total assets is non-negative.")

    if bool(row["current_assets_below_short_term_liabilities"]):
        observations.append("Current assets are below short-term liabilities.")
    else:
        observations.append("Current assets exceed short-term liabilities.")

    if bool(row["liabilities_exceed_assets"]):
        observations.append("Reported total liabilities exceed total assets.")
    else:
        observations.append("Reported total liabilities do not exceed total assets.")

    if bool(row["negative_equity"]):
        observations.append("Equity relative to total assets is negative.")

    return observations


def get_review_questions(row) -> list:

    questions = []

    if bool(row["negative_net_profit"]):
        questions.append("What actions are planned to restore profitability?")

    if bool(row["negative_ebit"]):
        questions.append("Can management explain the negative operating result?")

    if bool(row["negative_working_capital"]):
        questions.append("How will short-term working-capital requirements be funded?")

    if bool(row["current_assets_below_short_term_liabilities"]):
        questions.append("Is sufficient liquidity available to meet short-term obligations?")

    if bool(row["liabilities_exceed_assets"]):
        questions.append("Is additional collateral or shareholder support available?")

    if not questions:
        questions.append(
            "Confirm that the latest management accounts remain consistent "
            "with the submitted figures."
        )

    questions.append("Verify the requested facility amount and intended use of funds.")

    return questions


# ---------------------------------------------------------
# 5. Document builders
# ---------------------------------------------------------
def create_loan_application(row):

    application_date = "15 January 2026"
    document_id = f"{row['borrower_id']}_LOAN_APPLICATION_V1"

    story = [
        Paragraph("Corporate Loan Application", styles["DocumentTitle"]),
        paragraph(
            "Synthetic document created for a credit-risk "
            "decision-support demonstration.",
            "Disclaimer"
        ),
        Spacer(1, 8),
        Paragraph("Application details", styles["SectionHeading"]),
        create_table(
            [
                ["Field", "Value"],
                ["Document ID", document_id],
                ["Application date", application_date],
                ["Borrower ID", row["borrower_id"]],
                ["Applicant", row["company_name"]],
                ["Sector", row["sector"]],
                ["Country", row["country"]],
                ["Employees", row["employee_count"]],
            ],
            column_widths=[60 * mm, 110 * mm],
        ),
        Spacer(1, 10),
        Paragraph("Requested facility", styles["SectionHeading"]),
        create_table(
            [
                ["Field", "Value"],
                ["Requested amount", format_currency(row["requested_loan_amount_eur"])],
                ["Loan purpose", row["loan_purpose"]],
                ["Requested term", "36 months"],
                ["Repayment frequency", "Quarterly"],
            ],
            column_widths=[60 * mm, 110 * mm],
        ),
        PageBreak(),
        Paragraph("Business description", styles["SectionHeading"]),
        paragraph(
            f"{row['company_name']} is a synthetic "
            f"{str(row['sector']).lower()} company operating "
            f"in {row['country']}. The company reports "
            f"approximately {int(row['employee_count'])} employees."
        ),
        Spacer(1, 8),
        Paragraph("Purpose of financing", styles["SectionHeading"]),
        paragraph(
            f"The requested facility of "
            f"{format_currency(row['requested_loan_amount_eur'])} "
            f"will be used for {str(row['loan_purpose']).lower()}."
        ),
        Spacer(1, 8),
        Paragraph("Applicant declaration", styles["SectionHeading"]),
        paragraph(
            "The applicant confirms that the information supplied "
            "is complete to the best of its knowledge and may be "
            "used for a preliminary credit assessment."
        ),
        paragraph(
            "Company names, application details and narrative "
            "content in this document are synthetic.",
            "Disclaimer"
        ),
    ]

    source_text = (
        f"Corporate Loan Application. Document ID: {document_id}. "
        f"Borrower ID: {row['borrower_id']}. Applicant: {row['company_name']}. "
        f"Sector: {row['sector']}. Country: {row['country']}. "
        f"Employees: {row['employee_count']}. "
        f"Requested amount: {format_currency(row['requested_loan_amount_eur'])}. "
        f"Loan purpose: {row['loan_purpose']}. "
        f"Requested term: 36 months. Repayment frequency: quarterly."
    )

    return document_id, story, source_text


def create_financial_summary_doc(row):

    document_id = f"{row['borrower_id']}_FINANCIAL_SUMMARY_V1"

    ratio_rows = [
        ["Financial ratio", "Reported value"],
        ["Net profit / total assets", format_ratio(row["net_profit_to_total_assets"])],
        ["Total liabilities / total assets", format_ratio(row["total_liabilities_to_total_assets"])],
        ["Working capital / total assets", format_ratio(row["working_capital_to_total_assets"])],
        ["Current assets / short-term liabilities", format_ratio(row["current_assets_to_short_term_liabilities"])],
        ["Retained earnings / total assets", format_ratio(row["retained_earnings_to_total_assets"])],
        ["EBIT / total assets", format_ratio(row["ebit_to_total_assets"])],
        ["Equity / total assets", format_ratio(row["equity_to_total_assets"])],
        ["Sales / total assets", format_ratio(row["sales_to_total_assets"])],
        ["Operating profit margin", format_ratio(row["operating_profit_margin"])],
        ["Short-term liabilities / total assets", format_ratio(row["short_term_liabilities_to_total_assets"])],
    ]

    observations = get_objective_observations(row)

    story = [
        Paragraph("Financial Summary", styles["DocumentTitle"]),
        create_table(
            [
                ["Field", "Value"],
                ["Document ID", document_id],
                ["Borrower ID", row["borrower_id"]],
                ["Company", row["company_name"]],
                ["Reporting period", "Year ending 31 December 2025"],
                ["Financial-data source", row["financial_data_source"]],
            ],
            column_widths=[60 * mm, 110 * mm],
        ),
        Spacer(1, 10),
        Paragraph("Selected financial ratios", styles["SectionHeading"]),
        create_table(ratio_rows, column_widths=[110 * mm, 60 * mm]),
        PageBreak(),
        Paragraph("Objective financial observations", styles["SectionHeading"]),
    ]

    for observation in observations:
        story.append(Paragraph(f"\u2022 {safe_text(observation)}", styles["BodyText"]))
        story.append(Spacer(1, 4))

    story.extend([
        Spacer(1, 8),
        paragraph(
            "These observations describe the submitted financial ratios. "
            "They do not constitute a lending decision, credit rating or "
            "validated bank policy outcome.",
            "Disclaimer"
        )
    ])

    source_text = (
        f"Financial Summary for {row['company_name']}. "
        + " ".join([
            f"Net profit to total assets: {format_ratio(row['net_profit_to_total_assets'])}.",
            f"Total liabilities to total assets: {format_ratio(row['total_liabilities_to_total_assets'])}.",
            f"Working capital to total assets: {format_ratio(row['working_capital_to_total_assets'])}.",
            f"Current assets to short-term liabilities: {format_ratio(row['current_assets_to_short_term_liabilities'])}.",
            f"EBIT to total assets: {format_ratio(row['ebit_to_total_assets'])}.",
            f"Equity to total assets: {format_ratio(row['equity_to_total_assets'])}.",
        ])
        + " Objective observations: "
        + " ".join(observations)
    )

    return document_id, story, source_text


def create_credit_analyst_note(row):

    document_id = f"{row['borrower_id']}_ANALYST_NOTE_V1"
    observations = get_objective_observations(row)
    review_questions = get_review_questions(row)

    story = [
        Paragraph("Preliminary Credit Analyst Note", styles["DocumentTitle"]),
        create_table(
            [
                ["Field", "Value"],
                ["Document ID", document_id],
                ["Borrower ID", row["borrower_id"]],
                ["Company", row["company_name"]],
                ["Review status", "Preliminary review"],
                ["Decision status", "No decision recorded"],
            ],
            column_widths=[60 * mm, 110 * mm],
        ),
        Spacer(1, 10),
        Paragraph("Purpose", styles["SectionHeading"]),
        paragraph(
            "This note records objective review points from the submitted "
            "application and financial summary. It does not contain an "
            "approval, rejection or credit rating."
        ),
        Paragraph("Observed financial points", styles["SectionHeading"]),
    ]

    for observation in observations:
        story.append(Paragraph(f"\u2022 {safe_text(observation)}", styles["BodyText"]))
        story.append(Spacer(1, 4))

    story.extend([
        PageBreak(),
        Paragraph("Questions for further review", styles["SectionHeading"]),
    ])

    for question in review_questions:
        story.append(Paragraph(f"\u2022 {safe_text(question)}", styles["BodyText"]))
        story.append(Spacer(1, 5))

    story.extend([
        Spacer(1, 10),
        paragraph(
            "Human review is required before any lending decision. "
            "This synthetic note is intended only for demonstrating "
            "document retrieval and explainable AI workflows.",
            "Disclaimer"
        )
    ])

    source_text = (
        f"Preliminary Credit Analyst Note for {row['company_name']}. "
        f"No lending decision has been recorded. "
        f"Observed financial points: "
        + " ".join(observations)
        + " Questions for further review: "
        + " ".join(review_questions)
    )

    return document_id, story, source_text


# ---------------------------------------------------------
# 6. Write PDF helper
# ---------------------------------------------------------
def write_pdf(output_path: str, document_id: str, story: list):
    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=18 * mm,
        leftMargin=18 * mm,
        topMargin=18 * mm,
        bottomMargin=20 * mm,
        title=document_id,
        author="Synthetic Credit Risk Copilot Project",
    )
    doc.build(story, onFirstPage=create_footer(document_id), onLaterPages=create_footer(document_id))


# ---------------------------------------------------------
# 7. Generate all PDFs and manifest records
# ---------------------------------------------------------
manifest_records = []

document_builders = {
    "loan_application": create_loan_application,
    "financial_summary": create_financial_summary_doc,
    "credit_analyst_note": create_credit_analyst_note,
}

for _, row in document_facts.iterrows():

    borrower_id = row["borrower_id"]
    borrower_directory = os.path.join(DOCUMENT_ROOT, borrower_id)
    os.makedirs(borrower_directory, exist_ok=True)

    for document_type, builder in document_builders.items():

        document_id, story, source_text = builder(row)
        filename = f"{document_type}.pdf"
        output_path = os.path.join(borrower_directory, filename)

        write_pdf(output_path=output_path, document_id=document_id, story=story)

        manifest_records.append({
            "document_id": document_id,
            "borrower_id": borrower_id,
            "company_name": row["company_name"],
            "document_type": document_type,
            "filename": filename,
            "file_path": output_path,
            "file_extension": "pdf",
            "document_version": "1.0",
            "document_content_is_synthetic": True,
            "financial_features_are_synthetic": False,
            "source_text": source_text,
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        })

document_manifest = pd.DataFrame(manifest_records)

# ---------------------------------------------------------
# 8. Leakage and file validation
# ---------------------------------------------------------
forbidden_terms = [
    "bankrupt_within_1_year",
    "demo_risk_group",
    "selection_stress_score",
    "healthy_solvent",
    "stressed_solvent",
]

for forbidden_term in forbidden_terms:
    assert not document_manifest["source_text"].str.contains(
        forbidden_term, case=False, na=False, regex=False
    ).any(), f"Outcome leakage detected: {forbidden_term}"

assert len(document_manifest) == 90
assert document_manifest["document_id"].is_unique

missing_files = [
    fp for fp in document_manifest["file_path"] if not os.path.isfile(fp)
]
assert not missing_files, f"Missing PDF files: {missing_files[:5]}"

empty_files = [
    fp for fp in document_manifest["file_path"] if os.path.getsize(fp) == 0
]
assert not empty_files, f"Empty PDF files: {empty_files[:5]}"

# ---------------------------------------------------------
# 9. Save manifest
# ---------------------------------------------------------
document_manifest.to_parquet(DOCUMENT_MANIFEST_FILE, index=False)

# ---------------------------------------------------------
# 10. Final summary
# ---------------------------------------------------------
print(f"Document root    : {DOCUMENT_ROOT}")
print(f"Manifest         : {DOCUMENT_MANIFEST_FILE}")
print(f"Borrowers        : {document_manifest['borrower_id'].nunique()}")
print(f"PDF documents    : {len(document_manifest)}")
print(f"Missing files    : {len(missing_files)}")
print(f"Empty files      : {len(empty_files)}")
print("Outcome leakage  : none")

print("\nDocuments by type:")
print(document_manifest["document_type"].value_counts().to_string())

display(
    document_manifest[
        ["document_id", "borrower_id", "document_type", "file_path"]
    ].head(10)
)

In [0]:
# =============================================================
# BRONZE DOCUMENT INGESTION
#
# Reads the 90 PDFs as binary files and stores them in a
# governed Delta table with document metadata and checksums.
# =============================================================

from pyspark.sql import functions as F

DOCUMENT_ROOT = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/documents"
)

DOCUMENT_MANIFEST_FILE = (
    "/Volumes/demodatabrciks123/"
    "creditrisk/raw_file/structured/"
    "demo_cohort/document_manifest.parquet"
)

BRONZE_TABLE = (
    "demodatabrciks123.creditrisk."
    "bronze_credit_documents"
)

# ---------------------------------------------------------
# 1. Read all nested PDF files as binary records
# ---------------------------------------------------------
binary_documents_df = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.pdf")
    .load(DOCUMENT_ROOT)
)

# ---------------------------------------------------------
# 2. Normalize the Spark file paths
#
# Depending on the runtime, Spark might return:
#   dbfs:/Volumes/...
#   file:/Volumes/...
#   /Volumes/...
# ---------------------------------------------------------
binary_documents_df = (
    binary_documents_df
    .withColumn(
        "normalized_file_path",
        F.regexp_replace(
            F.col("path"),
            r"^(dbfs:|file:)",
            ""
        )
    )
    .withColumnRenamed(
        "modificationTime",
        "file_modification_time"
    )
    .withColumnRenamed(
        "length",
        "file_size_bytes"
    )
)

# ---------------------------------------------------------
# 3. Read the document manifest
# ---------------------------------------------------------
manifest_df = (
    spark.read
    .parquet(DOCUMENT_MANIFEST_FILE)
    .select(
        "document_id",
        "borrower_id",
        "company_name",
        "document_type",
        "filename",
        "file_path",
        "document_version",
        "document_content_is_synthetic",
        "financial_features_are_synthetic",
        "generated_at_utc",
    )
    .withColumnRenamed(
        "file_path",
        "manifest_file_path"
    )
)

# ---------------------------------------------------------
# 4. Join each PDF to its manifest record
# ---------------------------------------------------------
bronze_documents_df = (
    binary_documents_df
    .join(
        manifest_df,
        binary_documents_df["normalized_file_path"]
        == manifest_df["manifest_file_path"],
        how="left"
    )
    .withColumn(
        "content_sha256",
        F.sha2(F.col("content"), 256)
    )
    .withColumn(
        "file_extension",
        F.lower(
            F.regexp_extract(
                F.col("filename"),
                r"\.([^.]+)$",
                1
            )
        )
    )
    .withColumn(
        "mime_type",
        F.lit("application/pdf")
    )
    .withColumn(
        "pdf_header_hex",
        F.hex(
            F.substring(
                F.col("content"),
                1,
                4
            )
        )
    )
    .withColumn(
        "is_valid_pdf_header",
        F.col("pdf_header_hex") == F.lit("25504446")
    )
    .withColumn(
        "ingested_at_utc",
        F.current_timestamp()
    )
    .select(
        "document_id",
        "borrower_id",
        "company_name",
        "document_type",
        "document_version",
        "filename",
        F.col("normalized_file_path").alias("file_path"),
        "file_extension",
        "mime_type",
        "file_size_bytes",
        "file_modification_time",
        "content_sha256",
        "is_valid_pdf_header",
        "document_content_is_synthetic",
        "financial_features_are_synthetic",
        "generated_at_utc",
        "ingested_at_utc",
        "content",
    )
)

# ---------------------------------------------------------
# 5. Validate before writing
# ---------------------------------------------------------
total_documents = bronze_documents_df.count()

missing_manifest_records = (
    bronze_documents_df
    .filter(F.col("document_id").isNull())
    .count()
)

invalid_pdf_headers = (
    bronze_documents_df
    .filter(~F.col("is_valid_pdf_header"))
    .count()
)

duplicate_document_ids = (
    bronze_documents_df
    .groupBy("document_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_paths = (
    bronze_documents_df
    .groupBy("file_path")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

empty_documents = (
    bronze_documents_df
    .filter(F.col("file_size_bytes") <= 0)
    .count()
)

assert total_documents == 90, (
    f"Expected 90 documents, found {total_documents}"
)

assert missing_manifest_records == 0, (
    f"{missing_manifest_records} files did not match the manifest"
)

assert invalid_pdf_headers == 0, (
    f"{invalid_pdf_headers} files have invalid PDF headers"
)

assert duplicate_document_ids == 0, (
    f"{duplicate_document_ids} duplicate document IDs found"
)

assert duplicate_paths == 0, (
    f"{duplicate_paths} duplicate file paths found"
)

assert empty_documents == 0, (
    f"{empty_documents} empty documents found"
)

# ---------------------------------------------------------
# 6. Write the governed Bronze Delta table
# ---------------------------------------------------------
(
    bronze_documents_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

# ---------------------------------------------------------
# 7. Validate the saved table
# ---------------------------------------------------------
saved_df = spark.table(BRONZE_TABLE)

print(f"Bronze table            : {BRONZE_TABLE}")
print(f"Documents               : {saved_df.count()}")
print(
    f"Borrowers               : "
    f"{saved_df.select('borrower_id').distinct().count()}"
)
print(
    f"Unique document IDs     : "
    f"{saved_df.select('document_id').distinct().count()}"
)
print(f"Missing manifest records: {missing_manifest_records}")
print(f"Invalid PDF headers     : {invalid_pdf_headers}")
print(f"Empty documents         : {empty_documents}")

print("\nDocuments by type:")

display(
    saved_df
    .groupBy("document_type")
    .count()
    .orderBy("document_type")
)

display(
    saved_df.select(
        "document_id",
        "borrower_id",
        "document_type",
        "file_size_bytes",
        "content_sha256",
        "is_valid_pdf_header",
        "ingested_at_utc",
    ).limit(10)
)

In [0]:
# =============================================================
# SILVER DOCUMENT PARSING
#
# Input:  demodatabrciks123.creditrisk.bronze_credit_documents
# Output: demodatabrciks123.creditrisk.silver_credit_documents_parsed
#
# ai_parse_document v2.0:
#   - Pins the output schema (pages, elements, metadata, error_status)
#   - descriptionElementTypes="" disables AI figure descriptions
#     (these PDFs contain only text and simple tables)
#
# The binary content column is dropped here; it already lives
# in the Bronze table and there is no reason to store it twice.
# =============================================================

from pyspark.sql import functions as F

BRONZE_TABLE = (
    "demodatabrciks123.creditrisk."
    "bronze_credit_documents"
)

SILVER_TABLE = (
    "demodatabrciks123.creditrisk."
    "silver_credit_documents_parsed"
)

# ---------------------------------------------------------
# 1. Read Bronze documents
# ---------------------------------------------------------
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze documents: {bronze_df.count()}")
try:
    runtime_version = spark.conf.get(
        "spark.databricks.clusterUsageTags.sparkVersion"
    )
except Exception:
    runtime_version = spark.version  # serverless: use spark.version
print(f"Runtime: {runtime_version}")

# ---------------------------------------------------------
# 2. Parse each PDF
#
# ai_parse_document is a SQL function; F.expr() maps cleanly
# to the required MAP('version','2.0', ...) SQL type.
#
# version=2.0          : pins the VARIANT output schema
# descriptionElementTypes='' : disables AI figure captions
# ---------------------------------------------------------
silver_df = (
    bronze_df
    .withColumn(
        "parsed_document",
        F.expr(
            "ai_parse_document("
            "  content,"
            "  MAP("
            "    'version',               '2.0',"
            "    'descriptionElementTypes', ''"
            "  )"
            ")"
        )
    )
    .withColumn(
        "parsed_at_utc",
        F.current_timestamp()
    )
    # Binary content already exists in Bronze; avoid duplicating it
    .drop("content")
)

# ---------------------------------------------------------
# 3. Write once to prevent repeated parsing and cost
# ---------------------------------------------------------
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")

# ---------------------------------------------------------
# 4. Validate
# ---------------------------------------------------------
saved_silver = spark.table(SILVER_TABLE)
total = saved_silver.count()

# Check for parse errors surfaced in the VARIANT
parse_errors = (
    saved_silver
    .filter(
        F.expr("try_cast(parsed_document:error_status AS STRING) IS NOT NULL")
    )
    .count()
)

print(f"\nSilver rows         : {total}")
print(f"Parse errors        : {parse_errors}")
print(f"Columns             : {len(saved_silver.columns)}")

print("\nSample parsed_document keys (first row):")
display(
    saved_silver.select(
        "document_id",
        "borrower_id",
        "document_type",
        F.expr("parsed_document:metadata").alias("metadata"),
        F.expr("try_cast(parsed_document:error_status AS STRING)").alias("error_status"),
        "parsed_at_utc",
    ).limit(6)
)

In [0]:
# =============================================================
# SILVER VALIDATION
# =============================================================

saved_silver_df = spark.table(SILVER_TABLE)

document_count = saved_silver_df.count()

unique_document_count = (
    saved_silver_df
    .select("document_id")
    .distinct()
    .count()
)

unique_borrower_count = (
    saved_silver_df
    .select("borrower_id")
    .distinct()
    .count()
)

assert document_count == 90, (
    f"Expected 90 documents, found {document_count}"
)

assert unique_document_count == 90, (
    f"Expected 90 document IDs, found {unique_document_count}"
)

assert unique_borrower_count == 30, (
    f"Expected 30 borrowers, found {unique_borrower_count}"
)

print(f"Silver table        : {SILVER_TABLE}")
print(f"Documents           : {document_count}")
print(f"Unique documents    : {unique_document_count}")
print(f"Unique borrowers    : {unique_borrower_count}")

# =============================================================
# INSPECT PARSING ERRORS
# =============================================================

parse_errors_df = spark.sql(
    f"""
    SELECT
        document_id,
        borrower_id,
        document_type,
        parsed_document:error_status AS error_status
    FROM {SILVER_TABLE}
    WHERE CAST(parsed_document:error_status AS STRING)
          NOT IN ('[]', 'null')
    """
)

parse_error_count = parse_errors_df.count()

print(f"\nDocuments with parsing errors: {parse_error_count}")

if parse_error_count > 0:
    display(parse_errors_df)

# =============================================================
# FLATTEN AND INSPECT PARSED ELEMENTS
# =============================================================

parsed_elements_df = spark.sql(
    f"""
    SELECT
        d.document_id,
        d.borrower_id,
        d.document_type,

        element.pos AS element_position,

        element.value:id::INT
            AS element_id,

        element.value:type::STRING
            AS element_type,

        element.value:content::STRING
            AS element_content,

        element.value:confidence::DOUBLE
            AS extraction_confidence,

        element.value:bbox
            AS bounding_boxes

    FROM {SILVER_TABLE} AS d,
    LATERAL variant_explode(
        d.parsed_document:document:elements
    ) AS element
    """
)

element_count = parsed_elements_df.count()

documents_with_elements = (
    parsed_elements_df
    .select("document_id")
    .distinct()
    .count()
)

print(f"\nParsed elements       : {element_count}")
print(f"Documents with content: {documents_with_elements}")

assert documents_with_elements == 90, (
    f"Only {documents_with_elements} documents produced elements"
)

display(
    parsed_elements_df
    .select(
        "document_id",
        "document_type",
        "element_position",
        "element_type",
        "element_content",
        "extraction_confidence",
    )
    .orderBy(
        "document_id",
        "element_position"
    )
    .limit(30)
)

In [0]:
# =============================================================
# PAGE-AWARE RETRIEVAL CHUNKS — improved manual fallback
#
# ai_prep_search requires the workspace Preview to be enabled
# by an admin (Settings → Previews → AI Prep Search).
#
# Improvements over the first fallback:
# - page_id and page_number extracted from bbox metadata
# - Document title and section header carried into chunk_to_embed
# - Titles and section_headers excluded as standalone chunks
# - HTML kept in chunk_to_retrieve; stripped from chunk_to_embed
# - Change Data Feed enabled for future Delta Sync AI Search index
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

PARSED_TABLE = (
    "demodatabrciks123.creditrisk."
    "silver_credit_documents_parsed"
)

CHUNK_TABLE = (
    "demodatabrciks123.creditrisk."
    "silver_credit_document_chunks"
)

# ---------------------------------------------------------
# 1. Extract elements and their page metadata
# ---------------------------------------------------------
elements_df = spark.sql(
    f"""
    SELECT
        d.document_id,
        d.borrower_id,
        d.company_name,
        d.document_type,
        d.document_version,
        d.filename,
        d.file_path,
        d.mime_type,
        d.content_sha256,

        element.pos AS element_position,

        element.value:type::STRING
            AS element_type,

        element.value:content::STRING
            AS element_content,

        element.value:confidence::DOUBLE
            AS extraction_confidence,

        element.value:bbox[0]:page_id::INT
            AS page_id

    FROM {PARSED_TABLE} AS d,

    LATERAL variant_explode(
        d.parsed_document:document:elements
    ) AS element
    """
)

# Databricks page_id is zero-based; analyst-facing page numbers are one-based
elements_df = elements_df.withColumn(
    "page_number",
    F.col("page_id") + F.lit(1)
)

# ---------------------------------------------------------
# 2. Carry document title and latest section header forward
# ---------------------------------------------------------
running_window = (
    Window
    .partitionBy("document_id")
    .orderBy("element_position")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

document_window = (
    Window
    .partitionBy("document_id")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.unboundedFollowing
    )
)

elements_df = (
    elements_df
    .withColumn(
        "document_title",
        F.first(
            F.when(
                F.col("element_type") == "title",
                F.col("element_content")
            ),
            ignorenulls=True
        ).over(document_window)
    )
    .withColumn(
        "current_section",
        F.last(
            F.when(
                F.col("element_type") == "section_header",
                F.col("element_content")
            ),
            ignorenulls=True
        ).over(running_window)
    )
)

# ---------------------------------------------------------
# 3. Keep only actual evidence-bearing elements
#
# title and section_header remain as context but are not
# indexed as standalone retrieval results.
# ---------------------------------------------------------
content_df = (
    elements_df
    .filter(
        F.col("element_type").isin(
            "text",
            "table",
            "caption",
            "footnote"
        )
    )
    .filter(
        F.col("element_content").isNotNull()
        & (F.trim(F.col("element_content")) != "")
    )
)

# ---------------------------------------------------------
# 4. Produce clean embedding text
#
# Keep HTML in chunk_to_retrieve for faithful table evidence.
# Strip HTML tags and normalise whitespace for chunk_to_embed.
# ---------------------------------------------------------
content_df = (
    content_df
    .withColumn(
        "embedding_content",
        F.when(
            F.col("element_type") == "table",
            F.regexp_replace(
                F.col("element_content"),
                r"<[^>]+>",
                " "
            )
        ).otherwise(
            F.col("element_content")
        )
    )
    .withColumn(
        "embedding_content",
        F.regexp_replace(F.col("embedding_content"), "&nbsp;", " ")
    )
    .withColumn(
        "embedding_content",
        F.regexp_replace(F.col("embedding_content"), "&amp;", "&")
    )
    .withColumn(
        "embedding_content",
        F.trim(
            F.regexp_replace(F.col("embedding_content"), r"\s+", " ")
        )
    )
)

# ---------------------------------------------------------
# 5. Create final retrieval chunks
# ---------------------------------------------------------
chunks_df = (
    content_df
    .withColumn(
        "chunk_id",
        F.concat(
            F.col("document_id"),
            F.lit("_CHUNK_"),
            F.lpad(
                F.col("element_position").cast("string"),
                4,
                "0"
            )
        )
    )
    .withColumn(
        "databricks_chunk_id",
        F.lit(None).cast("string")
    )
    .withColumn("chunk_position", F.col("element_position"))
    .withColumn("chunk_to_retrieve", F.col("element_content"))
    .withColumn(
        "chunk_to_embed",
        F.concat_ws(
            " | ",
            F.concat(F.lit("Company: "),       F.col("company_name")),
            F.concat(F.lit("Document type: "),
                     F.regexp_replace(F.col("document_type"), "_", " ")),
            F.when(
                F.col("document_title").isNotNull(),
                F.concat(F.lit("Document title: "), F.col("document_title"))
            ),
            F.when(
                F.col("current_section").isNotNull(),
                F.concat(F.lit("Section: "), F.col("current_section"))
            ),
            F.when(
                F.col("page_number").isNotNull(),
                F.concat(F.lit("Page: "), F.col("page_number").cast("string"))
            ),
            F.col("embedding_content")
        )
    )
    .withColumn(
        "pages",
        F.array(
            F.struct(
                F.col("page_id").alias("page_id"),
                F.col("page_number").alias("page_number")
            )
        )
    )
    .withColumn("pages_json", F.to_json("pages"))
    .withColumn("chunked_at_utc", F.current_timestamp())
    .withColumn("retrieve_character_count", F.length("chunk_to_retrieve"))
    .withColumn("embed_character_count",    F.length("chunk_to_embed"))
    .withColumn(
        "is_empty_chunk",
        F.trim(
            F.coalesce(F.col("chunk_to_retrieve"), F.lit(""))
        ) == ""
    )
    .select(
        "chunk_id",
        "databricks_chunk_id",
        "document_id",
        "borrower_id",
        "company_name",
        "document_type",
        "document_version",
        "filename",
        "file_path",
        "mime_type",
        "content_sha256",
        "chunk_position",
        "element_type",
        "document_title",
        "current_section",
        "page_id",
        "page_number",
        "pages",
        "pages_json",
        "extraction_confidence",
        "chunk_to_retrieve",
        "chunk_to_embed",
        "chunked_at_utc",
        "retrieve_character_count",
        "embed_character_count",
        "is_empty_chunk",
    )
)

# ---------------------------------------------------------
# 6. Validate
# ---------------------------------------------------------
chunk_count = chunks_df.count()

documents_with_chunks = (
    chunks_df.select("document_id").distinct().count()
)

borrowers_with_chunks = (
    chunks_df.select("borrower_id").distinct().count()
)

duplicate_chunk_ids = (
    chunks_df.groupBy("chunk_id").count()
    .filter(F.col("count") > 1).count()
)

empty_chunks = (
    chunks_df.filter(F.col("is_empty_chunk")).count()
)

missing_embed_text = (
    chunks_df
    .filter(
        F.col("chunk_to_embed").isNull()
        | (F.trim(F.col("chunk_to_embed")) == "")
    ).count()
)

missing_page_numbers = (
    chunks_df.filter(F.col("page_number").isNull()).count()
)

standalone_headings = (
    chunks_df
    .filter(F.col("element_type").isin("title", "section_header"))
    .count()
)

assert chunk_count > 0
assert documents_with_chunks == 90
assert borrowers_with_chunks == 30
assert duplicate_chunk_ids == 0
assert empty_chunks == 0
assert missing_embed_text == 0
assert missing_page_numbers == 0
assert standalone_headings == 0

# ---------------------------------------------------------
# 7. Overwrite the chunk Delta table
# ---------------------------------------------------------
(
    chunks_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHUNK_TABLE)
)

# Enable Change Data Feed for the future Delta Sync index
spark.sql(
    f"""
    ALTER TABLE {CHUNK_TABLE}
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
    """
)

# ---------------------------------------------------------
# 8. Final validation
# ---------------------------------------------------------
saved_chunks = spark.table(CHUNK_TABLE)

print(f"Chunk table           : {CHUNK_TABLE}")
print(f"Chunks                : {saved_chunks.count()}")
print(f"Documents             : {documents_with_chunks}")
print(f"Borrowers             : {borrowers_with_chunks}")
print(f"Duplicate chunk IDs   : {duplicate_chunk_ids}")
print(f"Empty chunks          : {empty_chunks}")
print(f"Missing embed text    : {missing_embed_text}")
print(f"Missing page numbers  : {missing_page_numbers}")
print(f"Standalone headings   : {standalone_headings}")
print("Change Data Feed      : enabled")

# ---------------------------------------------------------
# 9. Inspect chunks
# ---------------------------------------------------------
display(
    saved_chunks
    .select(
        "chunk_id",
        "borrower_id",
        "document_type",
        "element_type",
        "page_number",
        "current_section",
        "chunk_to_retrieve",
        "chunk_to_embed",
        "pages_json",
    )
    .orderBy("document_id", "chunk_position")
    .limit(20)
)

# ---------------------------------------------------------
# 10. Chunk distribution by document type and element type
# ---------------------------------------------------------
display(
    saved_chunks
    .groupBy("document_type", "element_type")
    .agg(
        F.count("*").alias("chunk_count"),
        F.round(F.avg("retrieve_character_count"), 1).alias("avg_retrieve_chars"),
        F.round(F.avg("embed_character_count"),    1).alias("avg_embed_chars"),
    )
    .orderBy("document_type", "element_type")
)

In [0]:
# databricks-ai-search is not yet publicly available on PyPI.
# databricks-vectorsearch is the confirmed canonical SDK.
# It exposes VectorSearchClient with the same high-level API.
%pip install --upgrade databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
# =============================================================
# DATABRICKS AI SEARCH — ENDPOINT + DELTA SYNC INDEX
#
# Embedding model : databricks-gte-large-en
#   (only llm/v1/embeddings endpoint READY in this workspace;
#    swap to databricks-qwen3-embedding-0-6b if provisioned later)
#
# Source table    : silver_credit_document_chunks
#   delta.enableChangeDataFeed = true     (already set in cell 17)
#   PRIMARY KEY on chunk_id               (added below, step 0)
#
# Endpoint type   : STANDARD  (~15-20 min to reach ONLINE)
# Pipeline type   : TRIGGERED (economical; manual sync on demand)
# Query type      : HYBRID    (semantic + BM25 keyword)
# =============================================================

import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType
from databricks.vector_search.client import VectorSearchClient

CHUNK_TABLE        = "demodatabrciks123.creditrisk.silver_credit_document_chunks"
AI_SEARCH_ENDPOINT = "credit-risk-ai-search-endpoint"
AI_SEARCH_INDEX    = "demodatabrciks123.creditrisk.credit_document_chunks_index"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"

# ---------------------------------------------------------
# 0. Unity Catalog PRIMARY KEY constraint
#    Required by Delta Sync indexes before index creation.
#    In Delta Lake the constraint is informational (not enforced
#    at write time) but Vector Search checks for its presence.
# ---------------------------------------------------------
try:
    spark.sql(
        f"ALTER TABLE {CHUNK_TABLE} "
        f"ADD CONSTRAINT credit_doc_chunks_pk PRIMARY KEY (chunk_id)"
    )
    print("PRIMARY KEY constraint added to chunk_id")
except Exception as pk_err:
    print(f"PRIMARY KEY note : {str(pk_err)[:120]}")

# ---------------------------------------------------------
# 1. Create Standard AI Search endpoint (idempotent)
#    Standard endpoints take ~15-20 minutes to reach ONLINE.
# ---------------------------------------------------------
w = WorkspaceClient()

try:
    ep = w.vector_search_endpoints.get_endpoint(AI_SEARCH_ENDPOINT)
    print(f"Endpoint exists — state: {ep.endpoint_status.state}")
except Exception:
    print(f"Creating endpoint: {AI_SEARCH_ENDPOINT} ...")
    w.vector_search_endpoints.create_endpoint(
        name=AI_SEARCH_ENDPOINT,
        endpoint_type=EndpointType.STANDARD,
    )

# Poll until ONLINE (80 × 30 s = 40 min max)
print("Waiting for endpoint to reach ONLINE ...")
for _ in range(80):
    ep    = w.vector_search_endpoints.get_endpoint(AI_SEARCH_ENDPOINT)
    state = str(ep.endpoint_status.state)
    print(f"  {state}")
    if "ONLINE" in state.upper():
        print("Endpoint is ONLINE")
        break
    time.sleep(30)
else:
    raise TimeoutError(
        "Endpoint did not reach ONLINE within 40 minutes. "
        "Check the Vector Search UI for errors."
    )

# ---------------------------------------------------------
# 2. Create Delta Sync index (idempotent)
#
# chunk_id       → primary key   (implicit; do not add to columns_to_sync)
# chunk_to_embed → embedding src (implicit; do not add to columns_to_sync)
# columns_to_sync: all filterable / returnable metadata columns.
# ---------------------------------------------------------
COLUMNS_TO_SYNC = [
    "chunk_to_retrieve",
    "document_id",
    "borrower_id",
    "company_name",
    "document_type",
    "document_version",
    "filename",
    "file_path",
    "element_type",
    "current_section",
    "page_number",
    "pages_json",
    "extraction_confidence",
]

vsc = VectorSearchClient()

try:
    index = vsc.get_index(
        endpoint_name=AI_SEARCH_ENDPOINT,
        index_name=AI_SEARCH_INDEX,
    )
    print(f"Index already exists: {AI_SEARCH_INDEX}")
    print("Triggering incremental sync ...")
    index.sync()

except Exception as get_err:
    if any(k in str(get_err).lower() for k in ["does not exist", "not found", "404"]):
        print(f"Creating index: {AI_SEARCH_INDEX} ...")
        index = vsc.create_delta_sync_index(
            endpoint_name=AI_SEARCH_ENDPOINT,
            source_table_name=CHUNK_TABLE,
            index_name=AI_SEARCH_INDEX,
            pipeline_type="TRIGGERED",
            primary_key="chunk_id",
            embedding_source_column="chunk_to_embed",
            embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
            columns_to_sync=COLUMNS_TO_SYNC,
        )
        print("Index creation triggered — initial sync in progress")
    else:
        raise

# ---------------------------------------------------------
# 3. Wait for index to be ready and initial sync to complete
#    (120 × 30 s = 60 min max)
# ---------------------------------------------------------
print("Waiting for index to be ready ...")
for _ in range(120):
    idx     = w.vector_search_indexes.get_index(AI_SEARCH_INDEX)
    ready   = bool(idx.status and idx.status.ready)
    indexed = int(idx.status.indexed_row_count) if (idx.status and idx.status.indexed_row_count) else 0
    print(f"  ready={ready}  indexed_rows={indexed}")
    if ready:
        print(f"Index ready — {indexed} rows indexed")
        break
    time.sleep(30)
else:
    raise TimeoutError("Index did not become ready within 60 minutes.")

# ---------------------------------------------------------
# 4. Summary
# ---------------------------------------------------------
print(f"\nEndpoint  : {AI_SEARCH_ENDPOINT}")
print(f"Index     : {AI_SEARCH_INDEX}")
print(f"Embedding : {EMBEDDING_ENDPOINT}")
print(f"\nIndex description:")
print(index.describe())

In [0]:
# =============================================================
# AI SEARCH SMOKE TEST — HYBRID RETRIEVAL
#
# Critical: borrower_id filter prevents cross-borrower leakage.
# The copilot MUST always scope queries to the active borrower.
# =============================================================

from databricks.vector_search.client import VectorSearchClient

AI_SEARCH_ENDPOINT = "credit-risk-ai-search-endpoint"
AI_SEARCH_INDEX    = "demodatabrciks123.creditrisk.credit_document_chunks_index"
TEST_BORROWER_ID   = "B000187"

vsc   = VectorSearchClient()
index = vsc.get_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    index_name=AI_SEARCH_INDEX,
)

results = index.similarity_search(
    query_text=(
        "What are the main profitability, leverage "
        "and liquidity observations for this borrower?"
    ),
    columns=[
        "chunk_id",
        "borrower_id",
        "company_name",
        "document_id",
        "document_type",
        "current_section",
        "page_number",
        "chunk_to_retrieve",
    ],
    # Standard endpoint filters: dict format
    filters={"borrower_id": TEST_BORROWER_ID},
    num_results=8,
    query_type="HYBRID",
)

# ---------------------------------------------------------
# Parse and display results
# ---------------------------------------------------------
col_names  = [c["name"] for c in results["manifest"]["columns"]]
data_array = results["result"]["data_array"]

print(f"Query    : profitability / leverage / liquidity")
print(f"Borrower : {TEST_BORROWER_ID}")
print(f"Chunks   : {len(data_array)} returned\n")
print("-" * 72)

for row in data_array:
    row_dict = dict(zip(col_names, row))
    score    = row[-1]          # similarity score is always last
    content  = (row_dict.get("chunk_to_retrieve") or "")[:180]
    print(f"chunk_id      : {row_dict.get('chunk_id')}")
    print(f"document_type : {row_dict.get('document_type')}")
    print(f"section       : {row_dict.get('current_section')}")
    print(f"page          : {row_dict.get('page_number')}")
    print(f"score         : {score:.4f}")
    print(f"content       : {content}...")
    print("-" * 72)

In [0]:
# =============================================================
# GUARDED FINANCIAL-EVIDENCE RETRIEVER
#
# Enforces:
#   - borrower_id isolation (no cross-borrower leakage)
#   - document_type allowlist (financial_summary + analyst note)
#   - current_section allowlist (quantitative risk sections only)
#   - hybrid retrieval (semantic + BM25)
#   - page_number present on every returned chunk
#
# DatabricksReranker is installed (databricks-ai-search 0.75)
# but not yet enabled in this workspace. Uncomment the reranker
# block when workspace support is confirmed.
# =============================================================

import pandas as pd
from databricks.ai_search.client import AISearchClient

AI_SEARCH_ENDPOINT = "credit-risk-ai-search-endpoint"
AI_SEARCH_INDEX    = (
    "demodatabrciks123.creditrisk."
    "credit_document_chunks_index"
)

client = AISearchClient(disable_notice=True)
index  = client.get_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    index_name=AI_SEARCH_INDEX,
)

# Sections that contain quantitative credit-risk evidence.
# Any chunk whose current_section is not in this list is excluded.
FINANCIAL_SECTIONS = [
    "Selected financial ratios",
    "Objective financial observations",
    "Observed financial points",
]

# Only documents that carry financial analysis are searched.
FINANCIAL_DOCUMENT_TYPES = [
    "financial_summary",
    "credit_analyst_note",
]


def _response_to_df(response: dict) -> pd.DataFrame:
    """
    Convert an AISearchClient similarity_search response to a
    pandas DataFrame.  The SDK appends a relevance score column
    after the requested columns.
    """
    col_names  = [c["name"] for c in response["manifest"]["columns"]]
    data_array = response["result"]["data_array"]

    if not data_array:
        return pd.DataFrame(columns=col_names)

    extra = len(data_array[0]) - len(col_names)
    if extra == 1:
        col_names = col_names + ["score"]
    elif extra > 1:
        col_names = col_names + [f"extra_{i}" for i in range(extra)]

    return pd.DataFrame(data_array, columns=col_names)


def retrieve_financial_evidence(
    borrower_id: str,
    question: str,
    num_results: int = 6,
) -> pd.DataFrame:
    """
    Borrower-scoped financial evidence retriever.

    Returns up to num_results chunks from financial_summary and
    credit_analyst_note documents, restricted to the quantitative
    sections defined in FINANCIAL_SECTIONS.

    Rows are ranked by Reciprocal Rank Fusion (hybrid score).
    When DatabricksReranker is enabled, add the reranker= argument
    for an additional LLM-based re-scoring pass.

    Raises
    ------
    ValueError
        If borrower_id or question are invalid.
    AssertionError
        If any retrieved chunk violates the isolation guarantees.
    """
    if not borrower_id or not borrower_id.startswith("B"):
        raise ValueError(
            f"borrower_id must start with 'B', got: {borrower_id!r}"
        )
    if not question or not question.strip():
        raise ValueError("question must not be empty.")

    response = index.similarity_search(
        query_text=question,
        # Only columns present in COLUMNS_TO_SYNC (set at index creation)
        # plus the primary key are retrievable.
        # document_title and chunk_to_embed were not synced.
        columns=[
            "chunk_id",
            "borrower_id",
            "company_name",
            "document_id",
            "document_type",
            "current_section",
            "page_number",
            "element_type",
            "chunk_to_retrieve",
        ],
        # AND across keys; list value = match any element in list.
        filters={
            "borrower_id":    borrower_id,
            "document_type":  FINANCIAL_DOCUMENT_TYPES,
            "current_section": FINANCIAL_SECTIONS,
        },
        query_type="hybrid",
        # Uncomment when DatabricksReranker is enabled in this workspace:
        # reranker=DatabricksReranker(
        #     columns_to_rerank=["chunk_to_embed", "chunk_to_retrieve", "current_section"]
        # ),
        num_results=num_results,
    )

    df = _response_to_df(response)

    if df.empty:
        print(
            f"No chunks found for borrower {borrower_id} "
            "in the specified sections. "
            "Verify current_section values in the chunk table."
        )
        return df

    # ----------------------------------------------------------
    # Retrieval safety assertions
    # ----------------------------------------------------------
    assert (df["borrower_id"] == borrower_id).all(), \
        "Cross-borrower retrieval detected."
    assert df["document_type"].isin(FINANCIAL_DOCUMENT_TYPES).all(), \
        "Unexpected document_type in results."
    assert df["current_section"].isin(FINANCIAL_SECTIONS).all(), \
        "Unexpected section in results."
    assert df["page_number"].notna().all(), \
        "Retrieved chunk is missing page_number."

    return df


print("retrieve_financial_evidence() defined")
print(f"  Endpoint : {AI_SEARCH_ENDPOINT}")
print(f"  Index    : {AI_SEARCH_INDEX}")
print(f"  Sections : {FINANCIAL_SECTIONS}")
print(f"  Doc types: {FINANCIAL_DOCUMENT_TYPES}")

In [0]:
TEST_BORROWER_ID = "B000187"

TEST_QUESTION = """
Assess this borrower's profitability, leverage and liquidity.

Find evidence covering:
- net profit and EBIT,
- total liabilities relative to assets,
- working capital,
- current assets versus short-term liabilities,
- equity relative to assets.
"""

financial_results = retrieve_financial_evidence(
    borrower_id=TEST_BORROWER_ID,
    question=TEST_QUESTION,
    num_results=6,
)

print(f"Chunks returned: {len(financial_results)}")

if not financial_results.empty:
    display(
        financial_results[[
            "borrower_id",
            "document_type",
            "current_section",
            "page_number",
            "chunk_to_retrieve",
            "score",
        ]]
    )